# Module 05 Lab - Data Preparation**Objective:** To learn and apply the most common data preparation techniques. Raw data is rarely ready for a machine learning model. This process, also called preprocessing, is one of the most critical steps in the entire ML workflow.**In this lab, you will write more of the code.** Read the explanations and then complete the tasks in the code cells.

## Part 1: Setup and Initial LookWe will continue using the Titanic dataset because it has the exact problems we need to solve: missing values and non-numeric data.

In [7]:
import pandas as pd
import numpy as np
# Load the dataset
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
# Let's look at the missing values
print("--- Missing Values Before Cleaning ---")
print(df.isnull().sum())

--- Missing Values Before Cleaning ---
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


## Part 2: Handling Missing Values (Imputation)**Concept:** Most machine learning models cannot handle missing values (`NaN`). We must deal with them. Dropping the rows is an option, but you lose data. A better way is **imputation**, which means filling in the missing values with a calculated guess.Common imputation strategies:*   **Mean:** Fill with the average value. Good for normally distributed data.*   **Median:** Fill with the middle value. Better for skewed data or data with outliers (like `Fare`).*   **Mode:** Fill with the most frequent value. Used for categorical data.

### Task 1: Impute the 'Age' ColumnThe 'Age' column is missing many values. Since age can be skewed (e.g., by a few very old passengers), using the **median** is a robust choice.**Your Task:** Calculate the median of the 'Age' column and use the `.fillna()` method to replace the missing values.

In [6]:
# --- ENTER YOUR CODE HERE ---
# 1. Calculate the median of the 'Age' column
median_age = df['Age'].median()

# 2. Fill the missing values in 'Age' with the median
df['Age'] = df['Age'].fillna(median_age)

# 3. Verify that there are no more missing values in 'Age'
print("Missing values in 'Age' after imputation:")
print(df['Age'].isnull().sum())

Missing values in 'Age' after imputation:
0


## Part 3: Encoding Categorical Features**Concept:** Machine learning models are mathematical, so they need numbers, not text. We need to convert categorical columns (like 'Sex' and 'Embarked') into a numerical format. The most common method is **One-Hot Encoding**.One-Hot Encoding takes a column with `N` categories and turns it into `N` new columns, each with a `1` or `0`. For example, the 'Sex' column (`male`, `female`) becomes two new columns: `Sex_male` and `Sex_female`.Pandas has a convenient function called `pd.get_dummies()` that does this for us.

### Task 2: One-Hot Encode Categorical Columns**Your Task:** Use `pd.get_dummies()` to encode the 'Sex' and 'Embarked' columns. Make sure to drop the original columns after encoding.

In [5]:
# --- ENTER YOUR CODE HERE ---
# 1. Use get_dummies to create new columns for 'Sex' and 'Embarked'
#    Set `drop_first=True` to avoid multicollinearity (a statistical issue), which drops one of the new columns (e.g., just having `Sex_male` is enough to know if someone is female).

# Drop the 'Cabin' column as it has too many missing values and is difficult to impute meaningfully for this dataset
df.drop('Cabin', axis=1, inplace=True)

df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

# 2. Display the first few rows of the new DataFrame to see the new columns
print("DataFrame after One-Hot Encoding:")
print(df.head())

DataFrame after One-Hot Encoding:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name   Age  SibSp  Parch  \
0                            Braund, Mr. Owen Harris  22.0      1      0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  38.0      1      0   
2                             Heikkinen, Miss. Laina  26.0      0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  35.0      1      0   
4                           Allen, Mr. William Henry  35.0      0      0   

             Ticket     Fare  Sex_male  Embarked_Q  Embarked_S  
0         A/5 21171   7.2500      True       False        True  
1          PC 17599  71.2833     False       False       False  
2  STON/O2. 3101282   7.9250     False       False        True  
3            113803  53.1000    

## Part 4: Feature Scaling**Concept:** Many models are sensitive to the scale of the features. For example, `Age` (from 0-80) and `Fare` (from 0-512) are on very different scales. This can cause the model to incorrectly believe that `Fare` is a more important feature simply because its values are larger.**Feature Scaling** solves this by putting all features on a similar scale. A common method is **Standardization** (`StandardScaler` in scikit-learn), which rescales the data to have a mean of 0 and a standard deviation of 1.**Important:** You only scale your numerical features, not your target variable or your newly encoded categorical columns.

### Task 3: Scale the 'Age' and 'Fare' Columns**Your Task:** Use `StandardScaler` from `sklearn.preprocessing` to scale the 'Age' and 'Fare' columns.

In [9]:
from sklearn.preprocessing import StandardScaler

# --- ENTER YOUR CODE HERE ---
# 1. Create an instance of the StandardScaler
scaler = StandardScaler()

# 2. Select the columns to scale
columns_to_scale = ['Age', 'Fare']

# 3. Fit the scaler to the data and transform it
df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])

# 4. Display the first few rows to see the scaled data
print("DataFrame after Feature Scaling:")
print(df.head())

DataFrame after Feature Scaling:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex       Age  SibSp  \
0                            Braund, Mr. Owen Harris    male -0.530377      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  0.571831      1   
2                             Heikkinen, Miss. Laina  female -0.254825      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  0.365167      1   
4                           Allen, Mr. William Henry    male  0.365167      0   

   Parch            Ticket      Fare Cabin Embarked  
0      0         A/5 21171 -0.502445   NaN        S  
1      0          PC 17599  0.786845   C85        C  
2      0  STON/O2. 3101282 -0.488854   NaN        S  
3      0            113803  0.420730  C123     

## 📝 Knowledge Check**Instructions:** Answer the following questions in this markdown cell.1.  **Why is it often better to impute missing values with the median instead of the mean?**2.  **Explain in your own words what One-Hot Encoding does and why it is necessary.**3.  **Would you need to apply Feature Scaling to a Decision Tree model?** Why or why not? (Hint: Think about how a Decision Tree makes its splits).**[ENTER YOUR ANSWERS HERE]**

**Answers:**
1.  **Why is it often better to impute missing values with the median instead of the mean?**
    The median is a more robust measure of central tendency compared to the mean when the data contains outliers or is skewed. Outliers can significantly affect the mean, pulling it towards extreme values, which might lead to a less representative imputation. The median, being the middle value, is less sensitive to these extreme values, making it a safer choice for imputation in such cases.

2.  **Explain in your own words what One-Hot Encoding does and why it is necessary.**
    One-Hot Encoding converts categorical data (data with distinct categories, like 'gender' or 'city') into a numerical format that machine learning models can understand. It does this by creating new binary columns for each unique category in the original column. For example, if you have a 'Color' column with values 'Red', 'Green', 'Blue', One-Hot Encoding would create three new columns: 'Color_Red', 'Color_Green', and 'Color_Blue'. For each row, a '1' would be placed in the column corresponding to its original color, and '0's in the others. This is necessary because most machine learning algorithms require numerical input and cannot directly process text-based categories. It also prevents the model from assuming an arbitrary ordinal relationship between categories that doesn't exist (e.g., thinking 'Red' is 'greater than' 'Blue').

3.  **Would you need to apply Feature Scaling to a Decision Tree model?** Why or why not?
    No, you generally do not need to apply Feature Scaling to a Decision Tree model (or other tree-based models like Random Forests or Gradient Boosting Machines). Decision Trees make splits based on individual feature values and their thresholds. The splits are determined by finding the best value within a feature that separates the data, and this process is not affected by the scale of the feature values. For example, whether 'Age' is 22 or 2200, a split at 'Age > 30' will behave the same relative to the data points. Models that rely on distance calculations (like K-Nearest Neighbors, Support Vector Machines) or gradient descent (like Neural Networks, Logistic Regression) are sensitive to feature scales, but Decision Trees are not.

### Reflective Journal Guidelines for Data Preparation Notebooks

When writing a reflective journal entry, aim to capture not just *what* you did, but *why* you did it, *what you learned*, and *how it connects* to broader concepts.

**1. Identify the Notebook's Core Objective:**
   *   What was the main goal or learning objective of this notebook/lab? (e.g., learn data cleaning, practice feature engineering, understand a specific algorithm).

**2. Summarize Key Steps/Tasks Performed:**
   *   Briefly list the major actions you took in the notebook (e.g., loaded data, handled missing values, encoded categorical features, scaled numerical features).

**3. Detail What You Learned from Each Major Step:**
   *   For each key step, reflect on:
      *   **The Problem:** What issue was this step trying to solve? (e.g., missing values preventing model training, categorical data not being numerical).
      *   **The Solution/Technique:** What specific method was used? (e.g., median imputation, one-hot encoding, StandardScaler).
      *   **Why This Solution?** Why was this particular technique chosen over others? (e.g., median for skewed data, `drop_first=True` for multicollinearity, StandardScaler for its properties).
      *   **Key Insights/Best Practices:** What important concepts, nuances, or best practices did you learn or reinforce? (e.g., checking data distribution before imputation, the importance of `drop_first`, when not to scale for certain models).

**4. Reflect on Challenges and Solutions:**
   *   Did you encounter any errors or warnings? How did you resolve them? What did you learn from these challenges? (e.g., `FutureWarning` about `inplace=True`).
   *   Were there any parts that you found particularly difficult or surprisingly easy?

**5. Connect to Broader Concepts/Real-World Applications:**
   *   How do the techniques learned in this notebook apply to other data science projects or real-world scenarios?
   *   How does this preprocessing step fit into the overall machine learning pipeline?
   *   What is the impact of *not* performing these steps?

**6. Personal Thoughts and Future Implications:**
   *   What was your overall impression of the lab/notebook?
   *   What lingering questions do you have?
   *   How will this learning influence your approach to future data science tasks?
   *   Are there any alternative approaches you'd like to explore in the future?

By following these guidelines, your reflective journal will become a valuable resource for consolidating your learning and tracking your growth in data science.

### Reflective Journal: My First Steps in Data Prep

This notebook was helpful for understanding how to get data ready for machine learning. Prior to this, I did not fully consider all the necessary steps before model application. It appears raw data is often quite messy, as demonstrated by the Titanic dataset.

First, I learned that models generally do not process **missing values**, such as empty spots in the 'Age' and 'Cabin' columns. This necessitates filling these gaps. For 'Age', we used the **median** instead of the mean because the median is more robust if there are unusual age values that might skew the average. This approach helps ensure more accurate numerical representation.

Next, we addressed **categorical features**, which are descriptive categories rather than numerical values (like 'Sex' with 'male' or 'female', or 'Embarked' with 'S', 'C', 'Q'). Machine learning models require numerical input, so we needed to convert these. **One-Hot Encoding** was used, which converts each category into its own binary column, indicating presence with a '1' and absence with a '0'. This process translates categorical information into a format suitable for computation. I also learned about dropping one of the new columns (`drop_first=True`) to prevent multicollinearity, which can confuse the model.

Then came **Feature Scaling**. This step was interesting because I realized that some numerical features, like 'Age' (ranging from 0-80) and 'Fare' (ranging from 0-512), exist on vastly different scales. Without scaling, a model might incorrectly perceive 'Fare' as more important simply due to its larger numerical values. **StandardScaler** helped normalize 'Age' and 'Fare' to a similar scale, preventing the model from misinterpreting their relative importance.

The 'Cabin' column presented a challenge. It had so many missing values that attempting to fill them meaningfully was impractical. In such cases, it is sometimes best to remove a column if it is too incomplete.

The "Knowledge Check" section provided valuable insight into *why* these techniques are applied, beyond just *how* to implement them. For instance, I now understand that Decision Tree models do not typically require feature scaling because their splitting mechanism relies on direct value thresholds, not on distance metrics. This is a crucial detail to remember.

Overall, this lab highlighted that data preparation is a significant and essential component of the machine learning pipeline. It emphasizes that successful model building depends not only on advanced algorithms but also on providing them with clean and well-prepared data. This experience feels like acquiring fundamental tools for my data science journey.